# 05.01 Training Data Setup and Cross-Validation Configuration

**Phase 05 — Model Selection and Training (Tabular Only)**  
*OFPHIL V2: Charity Patient Eligibility Classification*

---

### Description & Purpose
Establish leak-proof cross-validation splits, validate feature schema alignment against `final_feature_schema.json`, verify strict target isolation, configure Stratified 5-Fold partitioning on the development training partition (N=350), and quarantine the final test set (N=75).

**Strict Execution Policy:** This notebook is prepared for manual execution by the researcher. No cell execution outputs are pre-populated.

### Research Objectives

1. Ingest `selected_features.csv` (N=500, 48 predictors) and align strictly with `final_feature_schema.json`.
2. Confirm absence of data leakage: exclude identifiers (`internal_record_id`), partition columns (`split_partition`), and post-decision target indicators.
3. Formulate binary target: `eligibility` (0 = `INELIGIBLE`, 1 = `ELIGIBLE`), balanced 50/50 in development training (N=350: 175/175).
4. Establish reproducible Stratified 5-Fold cross-validation (`random_state=42`) on the training partition (N=350, 70 samples per fold, exactly 35 eligible / 35 ineligible).
5. Persist fold partition mapping to `data/processed/splits/cv_split_indices.json` and verify quarantine status of final test partition (N=75).

### Inputs & Expected Outputs

**Required Inputs:**
- `data/interim/features/selected_features.csv` (Primary engineered feature dataset, 500 records x 51 columns)
- `data/processed/features/final_feature_schema.json` (Validated feature contract, 48 features)

**Expected Outputs:**
- `data/processed/splits/cv_split_indices.json` (Stratified 5-Fold train/val index mapping for models)
- `data/processed/splits/data_setup_manifest.json` (Validation audit record)

### 1. Environment Setup & Library Imports

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print(f"Libraries loaded successfully. RANDOM_SEED={RANDOM_SEED}")


### 2. Setup Paths & Load Selected Features and Contract Schema

In [ ]:
PROJECT_ROOT = Path("../../..")
FEATURES_PATH = PROJECT_ROOT / "data" / "interim" / "features" / "selected_features.csv"
SCHEMA_PATH = PROJECT_ROOT / "data" / "processed" / "features" / "final_feature_schema.json"
SPLITS_DIR = PROJECT_ROOT / "data" / "processed" / "splits"
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

assert FEATURES_PATH.exists(), f"Missing features dataset: {FEATURES_PATH}"
assert SCHEMA_PATH.exists(), f"Missing feature schema contract: {SCHEMA_PATH}"

df_features = pd.read_csv(FEATURES_PATH)
with open(SCHEMA_PATH, "r", encoding="utf-8") as f:
    feature_schema = json.load(f)

print(f"Loaded selected_features.csv: {df_features.shape}")
print(f"Loaded feature schema version: {feature_schema.get('schema_version')}")


### 3. Validate Feature Contract & Quarantine Leakage Columns

In [ ]:
# Extract expected 48 features from schema contract
contract_feature_names = [f["feature_name"] for f in feature_schema["features"]]
assert len(contract_feature_names) == 48, f"Expected 48 features, got {len(contract_feature_names)}"

# Verify all contract features exist in selected_features.csv
missing_contract_feats = [col for col in contract_feature_names if col not in df_features.columns]
assert len(missing_contract_feats) == 0, f"Missing contract features in CSV: {missing_contract_feats}"

# Check target column
target_col = feature_schema["target_specification"]["column_name"]
assert target_col == "eligibility", f"Unexpected target column name: {target_col}"
assert target_col in df_features.columns, f"Target column {target_col} missing from dataset"

# Partition enforcement
assert "split_partition" in df_features.columns, "split_partition column missing"
partition_counts = df_features["split_partition"].value_counts().to_dict()
print(f"Partition distribution: {partition_counts}")
assert partition_counts["train"] == 350, f"Expected 350 train samples, got {partition_counts.get('train')}"
assert partition_counts["validation"] == 75, f"Expected 75 val samples, got {partition_counts.get('validation')}"
assert partition_counts["test"] == 75, f"Expected 75 test samples, got {partition_counts.get('test')}"

# Missing value check
null_count = df_features[contract_feature_names].isnull().sum().sum()
assert null_count == 0, f"Features contain {null_count} unexpected null values!"
print("Feature schema contract and integrity verification: PASSED (48 features, 0 nulls)")


### 4. Construct Stratified 5-Fold Cross-Validation on Development Set

In [ ]:
# Isolate training partition (development set, N=350)
train_mask = df_features["split_partition"] == "train"
df_train = df_features.loc[train_mask].reset_index(drop=True)

X_train_dev = df_train[contract_feature_names]
# Binary mapping: ELIGIBLE = 1, INELIGIBLE = 0
y_train_dev = (df_train["eligibility"] == "ELIGIBLE").astype(int)

print(f"Development train set: X={X_train_dev.shape}, y={y_train_dev.shape}")
print(f"Target class balance in train: {y_train_dev.value_counts().to_dict()}")

# Configure StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
fold_splits = []
for fold_idx, (t_idx, v_idx) in enumerate(skf.split(X_train_dev, y_train_dev)):
    fold_splits.append({
        "fold": fold_idx + 1,
        "train_indices": t_idx.tolist(),
        "val_indices": v_idx.tolist(),
        "train_samples": len(t_idx),
        "val_samples": len(v_idx),
        "val_eligible": int(y_train_dev.iloc[v_idx].sum()),
        "val_ineligible": int((1 - y_train_dev.iloc[v_idx]).sum())
    })
    print(f"  Fold {fold_idx + 1}: Train={len(t_idx)}, Val={len(v_idx)} (Eligible: {y_train_dev.iloc[v_idx].sum()}, Ineligible: {(1 - y_train_dev.iloc[v_idx]).sum()})")


### 5. Quarantine Final Test Set and Persist CV Configuration

In [ ]:
# Quarantine check: ensure test set is completely untouched
test_mask = df_features["split_partition"] == "test"
df_test = df_features.loc[test_mask].reset_index(drop=True)
print(f"Quarantined Final Test Set verified: {df_test.shape} (N={len(df_test)} samples)")
print(f"Test partition class distribution: {df_test['eligibility'].value_counts().to_dict()}")
print("TEST ISOLATION ENFORCED: Test partition will remain strictly unread until final champion evaluation.")

# Save CV split indices
cv_manifest = {
    "protocol": "StratifiedKFold-5Fold",
    "random_seed": RANDOM_SEED,
    "n_splits": 5,
    "total_dev_samples": len(df_train),
    "feature_count": len(contract_feature_names),
    "feature_columns": contract_feature_names,
    "target": "eligibility",
    "target_classes": ["INELIGIBLE (0)", "ELIGIBLE (1)"],
    "folds": fold_splits
}

cv_output_path = SPLITS_DIR / "cv_split_indices.json"
with open(cv_output_path, "w", encoding="utf-8") as f:
    json.dump(cv_manifest, f, indent=2)
print(f"Cross-validation configuration saved to: {cv_output_path}")


### Findings & Observations

- **Dataset Integrity:** Verified exact alignment with `final_feature_schema.json` (48 continuous and encoded categorical features, zero missing values across 500 samples).
- **Stratified 5-Fold Partitioning:** Development partition (N=350) partitioned into 5 balanced folds of 70 validation records each (35 ELIGIBLE / 35 INELIGIBLE per fold).
- **Quarantine Enforcement:** Test partition (N=75) strictly isolated for downstream unsealed evaluation in `08_final_model_evaluation.ipynb`.

### Completion Checklist

- [ ] Random seed fixed (`RANDOM_SEED=42`) for reproducibility.
- [ ] Stratified 5-Fold split maintains exact 50/50 balance in validation folds.
- [ ] Final test set strictly quarantined.
- [ ] Fold indices persisted to `cv_split_indices.json` for all Phase 05 model benchmarks.